# Ноутбук 02 — Стационарность, АКФ, ЧАКФ, АДФ-тест
**Подраздел 2.3 ПЗ — Стационарность и корреляционные свойства ряда**

Запускать после `01_preprocessing_features.ipynb`.

Артефакты:
- `reports/figures/fig_2_8_acf_original.png` — Рисунок 2.8
- `reports/figures/fig_2_9_pacf_original.png` — Рисунок 2.9
- `reports/tables/table_2_2_adf_original.csv` — Таблица 2.2
- `data/interim/weekly_diff1.parquet` — дифференцированный недельный ряд
- `reports/figures/fig_2_10_acf_diff1.png` — Рисунок 2.10
- `reports/figures/fig_2_11_pacf_diff1.png` — Рисунок 2.11
- `reports/tables/table_2_3_seasonal_acf.csv` — Подтверждение S = 52

## Ячейка 0 — Импорты и конфигурация

In [ ]:
import sys
sys.path.insert(0, "..")
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from statsmodels.tsa.stattools import acf, pacf, adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

from src.config import (
    DATA_INT, FIGURES, TABLES,
    SEASONAL_PERIOD,
)

FIGURES.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

# Константы анализа
LAGS     = 104        # глубина АКФ / ЧАКФ — 2 года недель
ALPHA    = 0.05       # уровень значимости для ДИ АКФ и АДФ-теста
ADF_SPEC = "ct"      # спецификация: константа + тренд (p. 2.3.3)
SEASON   = SEASONAL_PERIOD  # 52 недели (из config.py)

# Настройки оформления графиков
plt.rcParams.update({
    "figure.dpi": 150,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 10,
})

print(f"SEASONAL_PERIOD : {SEASON}")
print(f"LAGS            : {LAGS}")
print(f"ADF_SPEC        : {ADF_SPEC}")

## Ячейка 1 — Загрузка недельного ряда

Источник: `data/interim/weekly_sales.parquet` (артефакт Ячейки 7 ноутбука 01).
Индекс приводится к частоте `W-MON`; если ряд непрерывный, выполнение прерывается с ошибкой.

In [ ]:
_raw = pd.read_parquet(DATA_INT / "weekly_sales.parquet")

# Столбец может называться 'sales' или быть единственным столбцом DataFrame
weekly = _raw.iloc[:, 0] if isinstance(_raw, pd.DataFrame) else _raw
weekly.index = pd.to_datetime(weekly.index)
weekly.name = "sales"

# Привод к явной частоте, заполнение пропусков интерполяцией (редкие выходные)
weekly = weekly.asfreq("W-MON", method="pad")

n_gaps = weekly.isna().sum()
assert n_gaps == 0, f"Пропуски в weekly_sales: {n_gaps}. Запустите ноутбук 01 заново."

print(f"Наблюдений : {len(weekly)}")
print(f"Период    : {weekly.index[0].date()} — {weekly.index[-1].date()}")
print(f"Пропусков  : {n_gaps}  — ОК")
print(f"Мин      : {weekly.min():,.0f}")
print(f"Макс      : {weekly.max():,.0f}")

## Ячейка 2 — АКФ исходного ряда (Рисунок 2.8)

Ожидаемые результаты:
- Медленное затухание АКФ — признак трендовой нестационарности (2.3.1).
- Выраженные пики на лагах 52 и 104 — сезонная структура с периодом S = 52.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
plot_acf(weekly, lags=LAGS, alpha=ALPHA, ax=ax, zero=False)

# Маркеры сезонных пиков
for s_lag in [SEASON, SEASON * 2]:
    ax.axvline(s_lag, color="tomato", linestyle="--",
               linewidth=1.0, alpha=0.8, label=f"Лаг {s_lag}")

ax.set_title("Рисунок 2.8 — АКФ недельного ряда суммарных продаж (исходный ряд)")
ax.set_xlabel("Лаг, нед.")
ax.set_ylabel("Автокорреляция")
ax.xaxis.set_major_locator(mticker.MultipleLocator(13))
ax.legend(loc="upper right")
fig.tight_layout()
fig.savefig(FIGURES / "fig_2_8_acf_original.png", dpi=150, bbox_inches="tight")
plt.show()
print("Рисунок 2.8 сохранён.")

## Ячейка 3 — ЧАКФ исходного ряда (Рисунок 2.9)

Метод оценки `ywm` (модифицированный Юл-Уокер) устойчив при наличии тренда и
не даёт отрицательных оценок ДИ, характерных для трендовых рядов.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
plot_pacf(weekly, lags=LAGS, alpha=ALPHA, ax=ax, zero=False, method="ywm")

for s_lag in [SEASON, SEASON * 2]:
    ax.axvline(s_lag, color="tomato", linestyle="--",
               linewidth=1.0, alpha=0.8, label=f"Лаг {s_lag}")

ax.set_title("Рисунок 2.9 — ЧАКФ недельного ряда суммарных продаж (исходный ряд)")
ax.set_xlabel("Лаг, нед.")
ax.set_ylabel("Частная автокорреляция")
ax.xaxis.set_major_locator(mticker.MultipleLocator(13))
ax.legend(loc="upper right")
fig.tight_layout()
fig.savefig(FIGURES / "fig_2_9_pacf_original.png", dpi=150, bbox_inches="tight")
plt.show()
print("Рисунок 2.9 сохранён.")

## Ячейка 4 — АДФ-тест исходного ряда (Таблица 2.2)

**Гипотезы:**
- H₀: в ряде присутствует единичный корень (ряд нестационарен)
- H₁: единичного корня нет (ряд стационарен)

Спецификация `ct` (константа + тренд) используется вследствие наличия
восходящего тренда и структурных сдвигов (2014‚2015, 2017).

In [ ]:
adf_out = adfuller(weekly, regression=ADF_SPEC, autolag="AIC")
stat, pvalue, lags_used, nobs = adf_out[0], adf_out[1], adf_out[2], adf_out[3]
crit_vals = adf_out[4]

table_22 = pd.DataFrame({
    "Показатель": [
        "Тестовая статистика",
        "p-значение",
        "Критическое значение 1 %",
        "Критическое значение 5 %",
        "Критическое значение 10 %",
        "Число лагов (AIC)",
        "Число наблюдений",
    ],
    "Значение": [
        round(stat, 4),
        round(pvalue, 4),
        round(crit_vals["1%"], 4),
        round(crit_vals["5%"], 4),
        round(crit_vals["10%"], 4),
        lags_used,
        nobs,
    ],
})

table_22.to_csv(TABLES / "table_2_2_adf_original.csv", index=False)
print("Таблица 2.2 сохранена.")
display(table_22)

if pvalue > ALPHA:
    print(f"\np = {pvalue:.4f} > {ALPHA} → H₀ не отвергается: ряд нестационарен → d = 1.")
else:
    print(f"\np = {pvalue:.4f} ≤ {ALPHA} → H₀ отвергается: ряд стационарен → d = 0.")

## Ячейка 5 — Определение d и первое дифференцирование (п. 2.3.3)

Параметр `d` принимается равным 1 при p-значении выше 0,05 и 0 в противном случае.
Дифференцированный ряд сохраняется в `data/interim/weekly_diff1.parquet` — 
источник для ноутбука 04 (САРИМА).

In [ ]:
d = 0 if pvalue <= ALPHA else 1
print(f"Порядок обычного дифференцирования: d = {d}")

if d >= 1:
    weekly_diff = weekly.diff(d).dropna()
    weekly_diff.name = f"sales_diff{d}"
    diff_path = DATA_INT / f"weekly_diff{d}.parquet"
    weekly_diff.to_frame().to_parquet(diff_path)
    print(f"Сохранено: {diff_path}  ({len(weekly_diff)} наблюдений)")
else:
    weekly_diff = weekly.copy()
    print("Дифференцирование не требуется (d = 0).")

# Подтверждение стационарности после дифференцирования
if d >= 1:
    adf_diff = adfuller(weekly_diff, regression="c", autolag="AIC")
    print(f"АДФ дифф. ряда: stat = {adf_diff[0]:.4f}, p = {adf_diff[1]:.4f}")
    status = "стационарен (ОК)" if adf_diff[1] <= ALPHA else "ВНИМАНИЕ: всё ещё нестационарен, рассмотрите d = 2"
    print(f"Дифф. ряд: {status}")

## Ячейка 6 — АКФ дифференцированного ряда (Рисунок 2.10)

Проверяем:
1. Быстрое затухание АКФ — тренд устранён.
2. Остающиеся значимые пики на лагах 52 и 104 — сезонная структура сохранена.
3. По числу значимых отсечений АКФ предварительно определяется порядок `q`.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
plot_acf(weekly_diff, lags=LAGS, alpha=ALPHA, ax=ax, zero=False)

for s_lag in [SEASON, SEASON * 2]:
    ax.axvline(s_lag, color="tomato", linestyle="--",
               linewidth=1.0, alpha=0.8, label=f"Лаг {s_lag}")

ax.set_title("Рисунок 2.10 — АКФ после первого дифференцирования")
ax.set_xlabel("Лаг, нед.")
ax.set_ylabel("Автокорреляция")
ax.xaxis.set_major_locator(mticker.MultipleLocator(13))
ax.legend(loc="upper right")
fig.tight_layout()
fig.savefig(FIGURES / "fig_2_10_acf_diff1.png", dpi=150, bbox_inches="tight")
plt.show()
print("Рисунок 2.10 сохранён.")

## Ячейка 7 — ЧАКФ дифференцированного ряда (Рисунок 2.11)

Число значимых отсечений ЧАКФ предварительно определяет порядок `p` несезонной
части САРИМА. Сезонный аналог (пики на лагах 52 и 104) передаётся в ЧАКФ-сезонную часть.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
plot_pacf(weekly_diff, lags=LAGS, alpha=ALPHA, ax=ax, zero=False, method="ywm")

for s_lag in [SEASON, SEASON * 2]:
    ax.axvline(s_lag, color="tomato", linestyle="--",
               linewidth=1.0, alpha=0.8, label=f"Лаг {s_lag}")

ax.set_title("Рисунок 2.11 — ЧАКФ после первого дифференцирования")
ax.set_xlabel("Лаг, нед.")
ax.set_ylabel("Частная автокорреляция")
ax.xaxis.set_major_locator(mticker.MultipleLocator(13))
ax.legend(loc="upper right")
fig.tight_layout()
fig.savefig(FIGURES / "fig_2_11_pacf_diff1.png", dpi=150, bbox_inches="tight")
plt.show()
print("Рисунок 2.11 сохранён.")

## Ячейка 8 — Подтверждение S = 52 (п. 2.3.5, Таблица 2.3)

АКФ считается на дифференцированном ряде: после устранения тренда
сезонные пики проявляются чище. Доверительные интервалы:
- `confint[i, 0]` и `confint[i, 1]` — абсолютные границы, значимость при
  `acf_val < confint[i, 0]` или `acf_val > confint[i, 1]`.

In [ ]:
_nlags = SEASON * 3  # 156 недель — проверяем три сезонных цикла
# для лага 156 достаточно наблюдений только при длине ряда >= 158
_available = len(weekly_diff) - 2  # максимальный лаг без потери надёжности
_nlags = min(_nlags, _available)

acf_vals, confint = acf(weekly_diff, nlags=_nlags, alpha=ALPHA)

season_lags = [l for l in [SEASON, SEASON * 2, SEASON * 3] if l <= _nlags]
rows = []
for lag in season_lags:
    val = acf_vals[lag]
    ci_lo = confint[lag, 0]
    ci_hi = confint[lag, 1]
    significant = bool(val < ci_lo or val > ci_hi)
    rows.append({
        "Лаг": lag,
        "АКФ": round(val, 4),
        "ДИ нижний": round(ci_lo, 4),
        "ДИ верхний": round(ci_hi, 4),
        "Значим": significant,
    })

season_df = pd.DataFrame(rows)
season_df.to_csv(TABLES / "table_2_3_seasonal_acf.csv", index=False)
print("Таблица 2.3 сохранена.")
display(season_df)

# Вывод решения для п. 2.3.5
sig_52 = season_df.loc[season_df["Лаг"] == SEASON, "Значим"].values
if len(sig_52) > 0 and sig_52[0]:
    print(f"\nЛаг {SEASON} значим → S = {SEASON} статистически подтверждён.")
else:
    print(f"\nЛаг {SEASON} незначим — проверьте исходный ряд или выбор S.")

## Ячейка 9 — Сводка параметров для САРИМА

In [ ]:
# Предварительная идентификация p и q по АКФ/ЧАКФ
# руководство: первые 2-3 значимых лага ЧАКФ → кандидаты p
# первые 2-3 значимых лага АКФ  → кандидаты q
# сезонные D, P, Q определяются по визуальному анализу рис. 2.10/2.11

print("=" * 55)
print("СВОДКА ПАРАМЕТРОВ САРИМА")
print("=" * 55)
print(f"  d = {d}   (обычное дифференцирование)")
print(f"  S = {SEASON}  (сезонный период, нед.)")
print(f"  p : кандидаты по ЧАКФ дифф. ряда (рис. 2.11)")
print(f"  q : кандидаты по АКФ  дифф. ряда (рис. 2.10)")
print(f"  D : 1, если лаги 52/104 значимы в АКФ дифф. ряда")
print(f"  P, Q: по сезонным отсечениям ЧАКФ/АКФ на лагах 52, 104")
print("=" * 55)
print("Запустите 04_econometric_sarima_holtwinters.ipynb")